In [2]:
pip install matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.1 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.1 MB 1.1 MB/s eta 0:00:07
   --- ------------------------------------ 0.8/8.1 MB 1.1 MB/s eta 0:00:07
   ----- ---------------------------------- 1.0/8.1 MB 1.1 MB/s eta 0:00:07
   ----- ---------------------------------- 1.0/8.1 MB 1.1 MB/s eta 0:00:07
   ------ --------------------------------- 1.3/8.1 MB 1.1 MB/s eta 0:00:07
   ------- -------------------------------- 1.6/8.1 MB 1.1 MB/s eta 0:00:06
   --------- ------------------------------ 1.8/8.1 MB 1.1 MB/s eta 0:00:06
   ---------- ----------------------------- 2.1/8.1 MB 1.1 MB/s eta 0:00:06
   ----------- ---------------------------- 2.4/8.1 MB 

In [4]:
pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
pip install lightgbm

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 1.1 MB/s eta 0:00:01
   --------------------- ------------------ 0.8/1.5 MB 1.1 MB/s eta 0:00:01
   ---------------------------- ----------- 1.0/1.5 MB 1.0 MB/s eta 0:00:01
   ------------------------------------ --- 1.3/1.5 MB 1.0 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 1.0 MB/s  0:00:01
Note: you may need to restart the kernel to use updated packages.


In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.base import BaseEstimator, RegressorMixin, clone
import xgboost as xgb
import lightgbm as lgb
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

In [8]:
class AdvancedFeatureEngineer:
    
    def __init__(self):
        self.component_cols = [f'Component{i}_fraction' for i in range(1, 6)]
        self.property_cols = [f'Component{i}_Property{j}' for i in range(1, 6) for j in range(1, 11)]
        self.target_cols = [f'BlendProperty{i}' for i in range(1, 11)]
    
    def create_interaction_features(self, df):
        interactions = pd.DataFrame(index=df.index)

        for i in range(5):
            for j in range(i+1, 5):
                col1 = f'Component{i+1}_fraction'
                col2 = f'Component{j+1}_fraction'
                interactions[f'{col1}_{col2}_interaction'] = df[col1] * df[col2]
        interactions['dominant_component'] = df[self.component_cols].max(axis=1)
        interactions['component_diversity'] = df[self.component_cols].std(axis=1)
        interactions['active_components'] = (df[self.component_cols] > 0.01).sum(axis=1)
        
        return interactions
    
    def create_weighted_properties(self, df):
        """Create fraction-weighted component properties"""
        weighted_props = pd.DataFrame(index=df.index)
        
        for prop_idx in range(1, 11):
            weighted_sum = 0
            for comp_idx in range(1, 6):
                fraction_col = f'Component{comp_idx}_fraction'
                property_col = f'Component{comp_idx}_Property{prop_idx}'
                weighted_sum += df[fraction_col] * df[property_col]
            
            weighted_props[f'WeightedProperty{prop_idx}'] = weighted_sum
        
        return weighted_props
    
    def create_statistical_features(self, df):
        """Create statistical features from component properties"""
        stats_features = pd.DataFrame(index=df.index)

        for prop_idx in range(1, 11):
            prop_cols = [f'Component{i}_Property{prop_idx}' for i in range(1, 6)]
            prop_data = df[prop_cols]
            
            stats_features[f'Property{prop_idx}_mean'] = prop_data.mean(axis=1)
            stats_features[f'Property{prop_idx}_std'] = prop_data.std(axis=1)
            stats_features[f'Property{prop_idx}_max'] = prop_data.max(axis=1)
            stats_features[f'Property{prop_idx}_min'] = prop_data.min(axis=1)
            stats_features[f'Property{prop_idx}_range'] = prop_data.max(axis=1) - prop_data.min(axis=1)

        for comp_idx in range(1, 6):
            comp_cols = [f'Component{comp_idx}_Property{j}' for j in range(1, 11)]
            comp_data = df[comp_cols]
            
            stats_features[f'Component{comp_idx}_prop_mean'] = comp_data.mean(axis=1)
            stats_features[f'Component{comp_idx}_prop_std'] = comp_data.std(axis=1)
        
        return stats_features
    
    def engineer_features(self, df):
        features = df[self.component_cols + self.property_cols].copy()
 
        interactions = self.create_interaction_features(df)
        weighted_props = self.create_weighted_properties(df)
        stats_features = self.create_statistical_features(df)

        engineered_df = pd.concat([features, interactions, weighted_props, stats_features], axis=1)
    
        engineered_df = engineered_df.fillna(0)
        
        return engineered_df

In [9]:
class StackingEnsemble(BaseEstimator, RegressorMixin):
    
    
    def __init__(self, base_models=None, meta_model=None, cv_folds=5, use_original_features=True):
        self.base_models = base_models if base_models else self._get_default_base_models()
        self.meta_model = meta_model if meta_model else Ridge(alpha=0.1)
        self.cv_folds = cv_folds
        self.use_original_features = use_original_features
        self.trained_base_models = []
        self.scaler = StandardScaler()
        
    def _get_default_base_models(self):
        return [
            ('rf', RandomForestRegressor(
                n_estimators=200, max_depth=15, min_samples_split=5,
                min_samples_leaf=2, random_state=42, n_jobs=-1
            )),
            ('xgb', xgb.XGBRegressor(
                n_estimators=200, max_depth=6, learning_rate=0.1,
                subsample=0.8, colsample_bytree=0.8, random_state=42
            )),
            ('lgb', lgb.LGBMRegressor(
                n_estimators=200, max_depth=6, learning_rate=0.1,
                subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1
            )),
            ('gb', GradientBoostingRegressor(
                n_estimators=150, max_depth=5, learning_rate=0.1,
                subsample=0.8, random_state=42
            )),
            ('et', ExtraTreesRegressor(
                n_estimators=150, max_depth=15, min_samples_split=5,
                min_samples_leaf=2, random_state=42, n_jobs=-1
            )),
            ('svr', SVR(kernel='rbf', C=1.0, gamma='scale')),
            ('elastic', ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42)),
            ('mlp', MLPRegressor(
                hidden_layer_sizes=(100, 50), max_iter=500, random_state=42
            ))
        ]
    
    def fit(self, X, y):

        X = np.array(X)
        y = np.array(y)
        
        X_scaled = self.scaler.fit_transform(X)

        kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=42)
        meta_features = np.zeros((X.shape[0], len(self.base_models)))
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(X_scaled)):
            X_train_fold, X_val_fold = X_scaled[train_idx], X_scaled[val_idx]
            y_train_fold = y[train_idx]
            
            for i, (name, model) in enumerate(self.base_models):
                
                model_clone = clone(model)
                model_clone.fit(X_train_fold, y_train_fold)
                
                val_preds = model_clone.predict(X_val_fold)
                meta_features[val_idx, i] = val_preds
        
        self.trained_base_models = []
        for name, model in self.base_models:
            model_clone = clone(model)
            model_clone.fit(X_scaled, y)
            self.trained_base_models.append((name, model_clone))
        
        if self.use_original_features:
            final_meta_features = np.column_stack([meta_features, X_scaled])
        else:
            final_meta_features = meta_features
        
        self.meta_model.fit(final_meta_features, y)
        
        return self
    
    def predict(self, X):
        X = np.array(X)
        X_scaled = self.scaler.transform(X)
        
        base_predictions = np.zeros((X.shape[0], len(self.trained_base_models)))
        
        for i, (name, model) in enumerate(self.trained_base_models):
            base_predictions[:, i] = model.predict(X_scaled)
        
        if self.use_original_features:
            final_meta_features = np.column_stack([base_predictions, X_scaled])
        else:
            final_meta_features = base_predictions
        
        return self.meta_model.predict(final_meta_features)

In [11]:
class FuelBlendMLPipeline:
    
    def __init__(self):
        self.feature_engineer = AdvancedFeatureEngineer()
        self.models = {}   
        self.scalers = {}  
        self.feature_names = None
        self.is_fitted = False
        
    def load_data(self, csv_path="train.csv"):
        df = pd.read_csv(csv_path)
        print(f"✅ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
        return df
    
    def preprocess(self, df):
        X = self.feature_engineer.engineer_features(df)
        self.feature_names = X.columns.tolist()
        
        y = df[self.feature_engineer.target_cols]
        
        return X, y
    
    def train(self, X, y, test_size=0.2, random_state=42):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=random_state
        )
        
        print(f"🔹 Training started on {X_train.shape[0]} samples, testing on {X_test.shape[0]}")
        
        for i, target in enumerate(self.feature_engineer.target_cols):
            print(f"\n🚀 Training model for {target}")
            
            model = StackingEnsemble(
                cv_folds=5,
                use_original_features=True
            )
            
            model.fit(X_train, y_train[target].values)
            
            preds = model.predict(X_test)
            mae = mean_absolute_error(y_test[target], preds)
            rmse = np.sqrt(mean_squared_error(y_test[target], preds))
            r2 = r2_score(y_test[target], preds)
            
            print(f"📊 {target} → MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")
            
            self.models[target] = model
        
        self.is_fitted = True
        print("\n✅ All models trained successfully.")
    
    def evaluate(self, X, y):
        if not self.is_fitted:
            raise Exception("❌ Models not trained yet. Call .train() first.")
        
        results = {}
        for target, model in self.models.items():
            preds = model.predict(X)
            mae = mean_absolute_error(y[target], preds)
            rmse = np.sqrt(mean_squared_error(y[target], preds))
            r2 = r2_score(y[target], preds)
            results[target] = {"MAE": mae, "RMSE": rmse, "R2": r2}
        
        return pd.DataFrame(results).T
    
    def predict(self, df):
        if not self.is_fitted:
            raise Exception("❌ Models not trained yet. Train before prediction.")
        
        X_new = self.feature_engineer.engineer_features(df)
        predictions = {}
        
        for target, model in self.models.items():
            predictions[target] = model.predict(X_new)
        
        return pd.DataFrame(predictions, index=df.index)
    
    def save_models(self, prefix="blend_model"):
        for target, model in self.models.items():
            joblib.dump(model, f"{prefix}_{target}.pkl")
        print("✅ All models saved.")
    
    def load_models(self, prefix="blend_model"):
        from glob import glob
        import os
        
        self.models = {}
        for target in self.feature_engineer.target_cols:
            path = f"{prefix}_{target}.pkl"
            if os.path.exists(path):
                self.models[target] = joblib.load(path)
        
        self.is_fitted = True
        print("✅ Models loaded successfully.")


In [12]:
pipeline = FuelBlendMLPipeline()

In [13]:
df = pipeline.load_data("train.csv")

✅ Data loaded: 2000 rows, 65 columns


In [14]:
X, y = pipeline.preprocess(df)

In [15]:
pipeline.train(X, y)

🔹 Training started on 1600 samples, testing on 400

🚀 Training model for BlendProperty1
📊 BlendProperty1 → MAE: 0.0599, RMSE: 0.0763, R²: 0.9937

🚀 Training model for BlendProperty2
📊 BlendProperty2 → MAE: 0.0741, RMSE: 0.0941, R²: 0.9904

🚀 Training model for BlendProperty3
📊 BlendProperty3 → MAE: 0.1274, RMSE: 0.1636, R²: 0.9720

🚀 Training model for BlendProperty4
📊 BlendProperty4 → MAE: 0.0591, RMSE: 0.0880, R²: 0.9921

🚀 Training model for BlendProperty5
📊 BlendProperty5 → MAE: 0.0462, RMSE: 0.0889, R²: 0.9926

🚀 Training model for BlendProperty6
📊 BlendProperty6 → MAE: 0.0482, RMSE: 0.0640, R²: 0.9958

🚀 Training model for BlendProperty7
📊 BlendProperty7 → MAE: 0.1303, RMSE: 0.1646, R²: 0.9718

🚀 Training model for BlendProperty8
📊 BlendProperty8 → MAE: 0.1078, RMSE: 0.1400, R²: 0.9797

🚀 Training model for BlendProperty9
📊 BlendProperty9 → MAE: 0.1848, RMSE: 0.2432, R²: 0.9439

🚀 Training model for BlendProperty10
📊 BlendProperty10 → MAE: 0.0650, RMSE: 0.0848, R²: 0.9930

✅ All 

In [16]:
eval_results = pipeline.evaluate(X, y)
print(eval_results)

                      MAE      RMSE        R2
BlendProperty1   0.051883  0.066869  0.995470
BlendProperty2   0.062820  0.080295  0.993607
BlendProperty3   0.095612  0.123456  0.984731
BlendProperty4   0.054406  0.073809  0.994648
BlendProperty5   0.032497  0.056433  0.996726
BlendProperty6   0.044089  0.057233  0.996782
BlendProperty7   0.098528  0.125948  0.984149
BlendProperty8   0.091292  0.117942  0.986048
BlendProperty9   0.133027  0.174391  0.969639
BlendProperty10  0.055584  0.071461  0.994792


In [17]:
sample_df = df.iloc[:5]

In [18]:
predictions = pipeline.predict(sample_df)
print("\n🔮 Predictions:\n", predictions.head())


🔮 Predictions:
    BlendProperty1  BlendProperty2  BlendProperty3  BlendProperty4  \
0        0.428469        0.636658        0.288998       -1.202031   
1       -1.258195       -1.380515       -0.514262       -1.441067   
2        1.859801        0.360989        0.644621        1.391670   
3       -0.102972        0.604720       -1.832529       -0.093808   
4       -0.015002       -1.125710        0.402653       -1.893904   

   BlendProperty5  BlendProperty6  BlendProperty7  BlendProperty8  \
0        1.568423        1.403044        0.278051        0.185296   
1        0.160061       -1.127997       -0.526071       -1.466725   
2       -0.452880        1.174276        0.639960        0.865468   
3       -0.142228       -0.238613       -1.811626        0.372165   
4       -0.475234       -0.582899        0.326462       -0.120849   

   BlendProperty9  BlendProperty10  
0        0.656751        -0.717884  
1       -1.288351        -0.590214  
2        0.717736         2.026326  
3    

In [20]:
import joblib

In [21]:
pipeline.save_models()

✅ All models saved.


In [23]:
model1 = joblib.load("blend_model_BlendProperty1.pkl")
model2 = joblib.load("blend_model_BlendProperty2.pkl")
model3 = joblib.load("blend_model_BlendProperty3.pkl")
model4 = joblib.load("blend_model_BlendProperty4.pkl")
model5 = joblib.load("blend_model_BlendProperty5.pkl")
model6 = joblib.load("blend_model_BlendProperty6.pkl")
model7 = joblib.load("blend_model_BlendProperty7.pkl")
model8 = joblib.load("blend_model_BlendProperty8.pkl")
model9 = joblib.load("blend_model_BlendProperty9.pkl")
model10 = joblib.load("blend_model_BlendProperty10.pkl")

In [24]:
blend_models = [model1, model2, model3, model4, model5, 
                model6, model7, model8, model9, model10]

In [70]:
from sklearn.model_selection import KFold
from sklearn.base import BaseEstimator, RegressorMixin, clone

In [94]:
class StackingRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, blend_models, meta_model, n_folds=5, random_state=None):
        self.blend_models = blend_models  
        self.meta_model = meta_model
        self.n_folds = n_folds
        self.random_state = random_state
        self.trained_base_models = []
        self.trained_meta_model = None

    def fit(self, X, y):
        X = pd.DataFrame(X)
        y = np.array(y).ravel()

        n_samples = X.shape[0]
        n_models = len(self.blend_models)
        meta_features = np.zeros((n_samples, n_models))

        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)

        for i, (name, model) in enumerate(self.blend_models):
            oof_preds = np.zeros(n_samples)

            for train_idx, val_idx in kf.split(X):
                X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
                y_tr, y_val = y[train_idx], y[val_idx]

                model_clone = clone(model)
                model_clone.fit(X_tr, y_tr)
                preds = model_clone.predict(X_val)

                oof_preds[val_idx] = preds

            meta_features[:, i] = oof_preds
            self.trained_base_models.append((name, clone(model).fit(X, y)))
        self.trained_meta_model = clone(self.meta_model)
        self.trained_meta_model.fit(meta_features, y)
        return self

    def predict(self, X):
        X = pd.DataFrame(X)
        base_preds = np.column_stack([
            model.predict(X) for _, model in self.trained_base_models
        ])
        return self.trained_meta_model.predict(base_preds)

In [95]:
blend_models_new = [
    ("rf", RandomForestRegressor(n_estimators=200, random_state=42)),
    ("gb", GradientBoostingRegressor(n_estimators=200, random_state=42))
]

In [96]:
meta_model = Ridge(alpha=1.0)

In [97]:
stacker = StackingRegressor(blend_models=blend_models_new, meta_model=meta_model, n_folds=5, random_state=42)


In [102]:
from sklearn.multioutput import MultiOutputRegressor

In [103]:
multi_stacker = MultiOutputRegressor(stacker)

In [104]:
multi_stacker.fit(X, y)

,estimator,StackingRegre...ndom_state=42)
,n_jobs,None
,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [105]:
print(multi_stacker.score(X, y)) 

0.9927209565299367


In [101]:
print(stacker.score(X, y["BlendProperty1"]))


0.994430265939248
